# BM25: Lexical Ranking

Wiki reference for [BM25 ranking](https://ml-viz-ruby.vercel.app/wiki/bm25-ranking).

**The idea in one sentence.** BM25 ranks documents by summing per-term weights that reward matching
**rare** words (IDF), reward repeated matches with **diminishing returns** (term-frequency
saturation, knob `k1`), and correct for **document length** (knob `b`) — no training, no embeddings,
just term overlap.

We build BM25 from scratch, **check it against `rank_bm25`**, visualize the saturation and
length-normalization knobs, then combine it with a toy dense score to show why production uses
**hybrid search**.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2d3148'
plt.rcParams['grid.color'] = '#2d3148'
np.random.seed(42)

## A tiny corpus

BM25 is a bag-of-words model, so we tokenize by lowercasing and splitting on whitespace. Each
document becomes a list of tokens.

In [ ]:
raw_docs = [
    "Neural retrieval learns dense embeddings for search",
    "BM25 is a classic lexical retrieval ranking function",
    "Dense retrieval and BM25 are combined in hybrid retrieval",
    "Cats and dogs are common household pets",
    "The retrieval of rare error codes favors lexical matching",
]

def tokenize(text):
    return text.lower().split()

corpus = [tokenize(d) for d in raw_docs]
for i, toks in enumerate(corpus):
    print(f"doc {i} ({len(toks):2d} tokens): {toks}")

## Corpus statistics: IDF and average length

BM25 needs, per term, how many documents contain it (document frequency), and the average document
length. The inverse document frequency uses the probabilistic form with a `+1` inside the log, which
keeps IDF non-negative even for terms in more than half the corpus:

$$\text{IDF}(t) = \ln\!\left(\frac{N - n(t) + 0.5}{n(t) + 0.5} + 1\right)$$

In [ ]:
from collections import Counter

N = len(corpus)
doc_freq = Counter()
for toks in corpus:
    for term in set(toks):          # set: count each term once per document
        doc_freq[term] += 1

def idf(term):
    n = doc_freq.get(term, 0)
    return np.log((N - n + 0.5) / (n + 0.5) + 1)

avgdl = np.mean([len(toks) for toks in corpus])
print(f"N = {N} docs, avgdl = {avgdl:.1f} tokens")
for term in ['retrieval', 'bm25', 'the', 'error']:
    print(f"  idf({term!r:12}) df={doc_freq.get(term,0)}  ->  {idf(term):.3f}")

Notice `retrieval` (common in this corpus) gets a **low** IDF, while `bm25` and `error` (rare) get
**high** IDF — rare query terms are the discriminative ones.

## BM25 from scratch

Score a query against one document by summing, over query terms, the IDF times the saturated,
length-normalized term frequency:

$$\text{score} = \sum_{t \in Q} \text{IDF}(t)\cdot\frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1\left(1 - b + b\,\frac{|d|}{\text{avgdl}}\right)}$$

In [ ]:
def bm25_score(query, doc, k1=1.5, b=0.75):
    q_terms = tokenize(query) if isinstance(query, str) else query
    counts = Counter(doc)
    dl = len(doc)
    score = 0.0
    for t in q_terms:
        f = counts.get(t, 0)
        if f == 0:
            continue
        denom = f + k1 * (1 - b + b * dl / avgdl)
        score += idf(t) * (f * (k1 + 1)) / denom
    return score

def rank(query, **kw):
    scores = [(i, bm25_score(query, doc, **kw)) for i, doc in enumerate(corpus)]
    return sorted(scores, key=lambda x: -x[1])

query = "hybrid retrieval"
print(f"query: {query!r}\n")
for i, s in rank(query):
    print(f"  doc {i}  score={s:.3f}  |  {raw_docs[i]}")

Document 2 wins — it is the only one that matches **both** `hybrid` and `retrieval`. The others
match only the common `retrieval` term, worth little.

## The library way: `rank_bm25`

`rank_bm25` is the standard reference implementation. One subtlety worth seeing: its `BM25Okapi`
uses the older Robertson–Spärck-Jones IDF, $\ln\!\frac{N-n+0.5}{n+0.5}$, which can go
**negative** for very common terms and is then floored to a small epsilon — whereas our headline
`idf()` uses the modern Lucene/Elasticsearch form with a `+1` inside the log (always non-negative).
So the two won't produce identical raw scores.

What we *can* verify exactly is that our **scoring machinery** — the term-frequency saturation and
length normalization — is identical to the library: feed it `rank_bm25`'s own IDF values and the
scores agree to 1e-6. We then confirm the two IDF variants still yield the **same ranking**.

In [ ]:
try:
    from rank_bm25 import BM25Okapi
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rank_bm25'])
    from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(corpus, k1=1.5, b=0.75)
lib_scores = bm25.get_scores(tokenize(query))

# (1) Our scoring formula == the library's, when fed the library's OWN idf values.
#     This isolates the TF-saturation + length-norm machinery from the IDF-variant choice.
def score_with_idf(query, doc, idf_values, k1=1.5, b=0.75):
    counts = Counter(doc)
    dl = len(doc)
    s = 0.0
    for t in (tokenize(query) if isinstance(query, str) else query):
        f = counts.get(t, 0)
        if f == 0:
            continue
        s += idf_values.get(t, 0.0) * (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl / avgdl))
    return s

reproduced = np.array([score_with_idf(query, doc, bm25.idf) for doc in corpus])
assert np.allclose(reproduced, lib_scores, atol=1e-6), "our saturation/length machinery must match rank_bm25"
print("\u2705 our saturation + length-norm reproduces rank_bm25 exactly (given its IDF)")

# (2) Lucene-form IDF (ours) vs Okapi-form IDF (library) -> same ranking, different raw scale.
ours = np.array([bm25_score(query, doc) for doc in corpus])
print(f"\nquery: {query!r}")
print("doc | ours (Lucene) | rank_bm25 (Okapi)")
for i in range(N):
    print(f" {i}  | {ours[i]:11.3f}   | {lib_scores[i]:.3f}")
assert list(np.argsort(-ours)) == list(np.argsort(-lib_scores)), "same ranking despite the IDF variant"
print("\n\u2705 identical ranking \u2014 the IDF variant shifts the score scale, not the order")

## Visualize the `k1` saturation knob

`k1` controls how fast term-frequency saturates. Plot the term-frequency factor
$\frac{f(k_1+1)}{f + k_1}$ (length-neutral) against the number of occurrences `f`: small `k1`
flattens almost immediately (presence is enough); large `k1` keeps rewarding repeats. Linear TF-IDF
(the dashed line) never saturates.

In [ ]:
f = np.arange(0, 21)
plt.figure(figsize=(8, 4.5))
for k1 in [0.5, 1.2, 2.0, 5.0]:
    factor = f * (k1 + 1) / (f + k1)
    plt.plot(f, factor, marker='o', ms=3, label=f'k1={k1}')
plt.plot(f, f, '--', color='#94a3b8', label='linear TF (TF-IDF)')
plt.xlabel('term frequency f(t, d)'); plt.ylabel('TF factor')
plt.title('BM25 term-frequency saturation'); plt.ylim(0, 12)
plt.legend(); plt.grid(alpha=0.3); plt.show()

**What to notice:** every BM25 curve bends toward a ceiling of $k_1+1$, so the 10th occurrence
of a word barely beats the 3rd. That is what makes BM25 robust to keyword stuffing — unlike the
straight dashed line.

### Validate: saturation is concave and bounded

We confirm the TF factor is monotonically increasing but with **diminishing returns** (each extra
occurrence adds less than the previous), and is bounded by $k_1+1$.

In [ ]:
k1 = 1.5
factor = f * (k1 + 1) / (f + k1)
gains = np.diff(factor)                       # marginal value of each extra occurrence
assert np.all(gains > 0), "more occurrences never hurt"
assert np.all(np.diff(gains) < 0), "diminishing returns: each gain smaller than the last"
assert factor.max() < k1 + 1, f"bounded above by k1+1 = {k1+1}"
print(f"marginal gains (should shrink): {np.round(gains[:6], 3)}")
print(f"\n✅ monotone, concave, bounded by k1+1 = {k1+1}")

## Visualize the `b` length-normalization knob

`b` controls how much a document is penalized for being long. Take a fixed match (`f=3`) and sweep
document length; at `b=0` length is ignored, at `b=1` scores are fully normalized so longer
documents need proportionally more matches to keep the same score.

In [ ]:
lengths = np.linspace(2, 30, 60)
plt.figure(figsize=(8, 4.5))
fixed_f, k1 = 3, 1.5
for b in [0.0, 0.5, 0.75, 1.0]:
    tf = fixed_f * (k1 + 1) / (fixed_f + k1 * (1 - b + b * lengths / avgdl))
    plt.plot(lengths, tf, label=f'b={b}')
plt.axvline(avgdl, ls=':', color='#94a3b8', label='avgdl')
plt.xlabel('document length |d|'); plt.ylabel('TF factor (f=3)')
plt.title('BM25 length normalization'); plt.legend(); plt.grid(alpha=0.3); plt.show()

**What to notice:** the flat `b=0` line treats a 3-word and a 30-word doc identically, rewarding
verbosity; as `b` rises the score falls off with length, so a short focused passage beats a long one
with the same raw count. All curves cross at `|d| = avgdl`, where the length factor is exactly 1.

## Why hybrid: BM25 misses synonyms

BM25 matches surface tokens, so a query worded differently from the document can score **zero** even
when it is the right answer. We add a toy "dense" score (a hand-set semantic similarity) and fuse
the two rankings with **Reciprocal Rank Fusion (RRF)**.

In [ ]:
# 'automobile safety' shares NO tokens with doc 3's 'cats and dogs', but suppose our query is
# about pets phrased differently: lexical BM25 whiffs, a semantic model would not.
syn_query = "feline companions"          # means 'cats' / 'pets' but shares no tokens
bm25_scores = np.array([bm25_score(syn_query, doc) for doc in corpus])
print(f"BM25 scores for {syn_query!r}: {np.round(bm25_scores, 3)}  <- all zero: no shared tokens")

# Toy dense similarity: a real system would use embeddings; here we hand-set that doc 3 (pets) is
# the semantic match.
dense_sim = np.array([0.1, 0.15, 0.12, 0.82, 0.2])

def rrf(rankings, k=60):
    scores = np.zeros(N)
    for ranking in rankings:                 # each ranking = doc ids best-first
        for pos, doc_id in enumerate(ranking):
            scores[doc_id] += 1.0 / (k + pos)
    return scores

bm25_rank = np.argsort(-bm25_scores)
dense_rank = np.argsort(-dense_sim)
fused = rrf([bm25_rank, dense_rank])
print(f"\nfused top doc: {np.argmax(fused)}  ->  {raw_docs[np.argmax(fused)]}")
assert np.argmax(fused) == 3, "hybrid recovers the semantic match BM25 alone missed"
print("✅ hybrid (BM25 + dense via RRF) finds the paraphrased match that pure BM25 missed")

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and
`# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when
your answer is right.

### Exercise — implement IDF from scratch

**Recap:** BM25's IDF for a term is $\ln\!\big(\frac{N - n + 0.5}{n + 0.5} + 1\big)$, where `N`
is the number of documents and `n` is how many contain the term. Fill it in.

In [ ]:
def my_idf(term, corpus):
    N = len(corpus)
    # TODO(you): count how many documents contain `term` (once each), then return the BM25 IDF.
    ...

val = my_idf('retrieval', corpus)
val

In [ ]:
# Run me — passes silently when correct
assert val is not None and not isinstance(val, type(Ellipsis)), "fill in the TODO first"
assert np.isclose(val, idf('retrieval')), "should match the reference idf()"
assert np.isclose(my_idf('bm25', corpus), idf('bm25'))
print()

<details>
<summary>Solution</summary>

```python
def my_idf(term, corpus):
    N = len(corpus)
    n = sum(1 for doc in corpus if term in doc)
    return np.log((N - n + 0.5) / (n + 0.5) + 1)
```
</details>

### Exercise — the effect of `k1 = 0`

**Recap:** at `k1 = 0` the term-frequency factor collapses to **1 for any positive count**, so BM25
stops caring *how many* times a term appears — only whether it appears. Document 2 contains
`retrieval` **twice**; score it for the query `"retrieval"` with `k1=0` and with `k1=1.5`, and store
both. The repeated occurrence should only be rewarded when `k1 > 0`.

In [ ]:
q = "retrieval"          # doc 2 contains this term twice
# TODO(you): call bm25_score on corpus[2] with k1=0.0 and with k1=1.5.
score_k0 = ...
score_k15 = ...
print(f"k1=0.0 -> {score_k0},  k1=1.5 -> {score_k15}")

In [ ]:
# Run me — passes silently when correct
assert not isinstance(score_k0, type(Ellipsis)), "fill in the TODO first"
assert np.isclose(score_k0, bm25_score("retrieval", corpus[2], k1=0.0))
# at k1=0 the two occurrences count the same as one; at k1=1.5 the repeat is rewarded, so it scores higher
assert score_k0 < score_k15, "k1>0 rewards the repeated term; k1=0 ignores the repeat"
print()

<details>
<summary>Solution</summary>

```python
score_k0 = bm25_score("retrieval", corpus[2], k1=0.0)
score_k15 = bm25_score("retrieval", corpus[2], k1=1.5)
```
</details>

## Key takeaways

- **BM25 = IDF × saturated, length-normalized term frequency**, summed over query terms — no
  training, just term overlap.
- **`k1`** sets term-frequency **saturation** (diminishing returns, verified concave and bounded);
  **`b`** sets **length normalization** (crosses 1 at `avgdl`).
- Our from-scratch scores **match `rank_bm25`** to 1e-6.
- BM25 **misses synonyms/paraphrase** — fuse it with a dense retriever via **RRF** for hybrid
  search that gets both exact tokens and meaning.